# Measurement Session Example

This notebook shows basic usage of `qubex.measurement.Measurement`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from qxpulse import Blank, FlatTop, Gaussian, PulseSchedule

from qubex.measurement import Measurement

In [ ]:
# Initialize Measurement
session = Measurement(
    chip_id="64Qv3",
    qubits=["Q62", "Q63"],
    config_dir="/home/shared/qubex-config/64Qv3/config",
    params_dir="/home/shared/qubex-config/64Qv3/params",
)

In [ ]:
Q0 = session.qubits[0]
Q1 = session.qubits[1]

In [ ]:
# Connect to devices (requires hardware access).
session.connect()

In [ ]:
# Measure noise on all qubits for 10,000 ns.
result = session.measure_noise(
    session.qubits,
    duration=10000,
)
result.plot()
result.data

In [ ]:
# Basic measurement with custom control waveforms
control_waveforms = {
    Q0: np.full(16, 0.03),
    Q1: np.full(16, 0.03),
}

# `measure` will automatically add readout pulses at the end of the sequence.

# Single-shot measurement
result = session.measure(
    control_waveforms,
    mode="single",
    shots=1024,
    plot=True,
)
result.plot()

In [ ]:
# Ensemble-averaged measurement
result = session.measure(
    control_waveforms,
    mode="avg",
    shots=1024,
)
result.plot()

In [ ]:
control_waveforms = {
    Q0: np.full(16, 0.03),
    Q1: np.full(16, 0.03),
}

result = session.measure(
    control_waveforms,
    mode="avg",
    shots=1024,
    readout_duration=1024,
    readout_ramptime=128,
    readout_pre_margin=256,
    readout_post_margin=512,
    plot=True,
)
result.plot()

In [ ]:
# Measurement from PulseSchedule
with PulseSchedule() as seq1:
    seq1.add("Q62", Gaussian(duration=32, amplitude=0.03, sigma=8))
    seq1.add("Q63", Gaussian(duration=32, amplitude=0.03, sigma=8))
    seq1.add("Q62", Blank(duration=16))
    seq1.barrier()
    seq1.add("RQ62", FlatTop(duration=256, amplitude=0.1, tau=32))
    seq1.add("RQ63", FlatTop(duration=256, amplitude=0.1, tau=32))
seq1.plot()

result = session.execute(
    seq1,
    mode="single",
    shots=1024,
)
result.plot()

In [ ]:
with PulseSchedule() as seq2:
    for _ in range(3):
        seq2.call(seq1)
seq2.plot()

result = session.execute(
    seq2,
    mode="single",
    shots=1024,
    plot=True,
    add_pump_pulses=True,
)
result.plot()

In [ ]:
schedule = session.build_measurement_schedule(
    pulse_schedule=seq1,
)

config = session.create_measurement_config(
    mode="single",
    shots=1024,
)

result = await session.run_measurement(
    schedule=schedule,
    config=config,
)
result